# **Notebook 04 — Predictive Modelling**

## **1. Business Question**

**Can the fulfilment problem be predicted early enough to act on?**

This notebook builds on the findings from Notebooks 02 and 03. The statistical analysis established that delivery lateness is strongly associated with poor customer reviews (**adjusted OR = 7.7, 95% CI [7.1, 8.4]**).

The predictive question is therefore more operational:

> **Can fulfilment risk be identified early enough for the business to intervene before the customer experience deteriorates?**

---

## **2. Analytical Chain**

The project follows the relationship established across the earlier notebooks:

```text
Geographic / Seller Structure
            ↓
     Fulfilment Burden
            ↓
 Delivery-Time Estimation
            ↓
      Late Deliveries
            ↓
       Poor Reviews
            ↓
       Intervention

## **1 · Setup**

In [ ]:
# Install Required Packages

!pip install scikit-learn shap --quiet


# Google Colab Authentication

from google.colab import auth

auth.authenticate_user()

# Import Libraries

import warnings

warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from google.cloud import bigquery

from sklearn.ensemble import (
    HistGradientBoostingRegressor,
    HistGradientBoostingClassifier
)

from sklearn.linear_model import (
    Ridge,
    LogisticRegression
)

from sklearn.preprocessing import StandardScaler

from sklearn.inspection import permutation_importance

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    average_precision_score,
    roc_auc_score,
    brier_score_loss
)

# BigQuery Configuration


PROJECT = "YOUR_PROJECT_ID"
DATASET = "olist"

client = bigquery.Client(
    project=PROJECT
)


# BigQuery Query Helper

def q(sql):
    return client.query(
        sql.replace(
            "`olist.",
            f"`{PROJECT}.{DATASET}."
        )
    ).to_dataframe()


# Random State

RANDOM_STATE = 0
print("ready")

---

## **2 · Leakage Audit**

Before engineering any features, every candidate variable is classified by **when the information becomes knowable**.

The objective is to ensure that the model only uses information that would genuinely be available at the time the prediction is made. Any feature that is created or revealed **after the delivery outcome occurs** is excluded, even if it improves validation performance.

This is the most important methodological safeguard in the notebook—and the section most likely to be challenged in an interview—because a model that uses future information can appear highly accurate while being unusable in practice.

**Prediction rule:** features must be available **at or before the prediction point**; the delivery outcome itself and any downstream information are never allowed into the feature set.

| Column | Known at checkout? | Decision |
|---|---|---|
| `purchase_ts`, month, weekday | Yes | **Use** |
| `item_value`, `freight_value`, `n_items`, `total_weight_g` | Yes | **Use** |
| `category` | Yes | **Use** |
| `customer_state`, `dominant_seller_state`, `max_distance_km` | Yes | **Use** |
| `promised_days` (Olist's own estimate) | Yes | **Use with care** — see below |
| Seller's *prior* delivery performance | Yes, if strictly backward-looking | **Use with a rolling window** |
| `approved_ts`, `carrier_ts` | **No** — post-purchase | **Exclude** |
| `delivered_ts`, `delivery_days` | **No** — this is the target | **Exclude** |
| `lateness_days`, `late_flag` | **No** — derived from the target | **Exclude** |
| `review_score`, `poor_review` | **No** — post-delivery | **Exclude** |
| Seller's *full-period* averages | **No** — contain future orders | **Exclude** |

**On `promised_days`.** Olist's own delivery estimate is available at checkout, so including it is **not data leakage**. However, it changes the interpretation of the model: the model can partly learn when Olist itself assigns a tighter or looser delivery window and how often those estimates are subsequently missed. The **primary model therefore excludes `promised_days`**, keeping the prediction independent of Olist's existing estimate. A sensitivity model including it is reported separately in §7.

**The rolling-window trap.** A seller's average delivery performance must never be calculated over the full dataset, because that would allow future orders to influence predictions for earlier orders. For each order, seller-history features must use **only orders placed before that order**, implemented in SQL with:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING

---

# **3 · Feature Table**

The modelling feature table is built in **BigQuery** so that the temporal ordering and no-leakage logic are explicit, reproducible, and auditable rather than hidden inside pandas transformations.

The key safeguard is the **window frame used for seller-history features**:

```sql
ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING

In [ ]:

# Create ML Feature View — Delivery Prediction


q("""
    CREATE OR REPLACE VIEW `olist.v_ml_delivery_features` AS

    WITH base AS (
        SELECT
            order_id,
            dominant_seller_id,
            purchase_ts,
            purchase_date,
            delivery_days,
            promised_days,          -- targets / benchmark, dropped before fitting
            item_value,
            freight_value,
            n_items,
            total_weight_g,
            category,
            customer_state,
            dominant_seller_state,
            max_distance_km,

            EXTRACT(
                MONTH FROM purchase_ts
            ) AS month,

            EXTRACT(
                DAYOFWEEK FROM purchase_ts
            ) AS dow

        FROM `olist.v_fact_orders`

        WHERE order_status = 'delivered'
          AND delivery_days IS NOT NULL
          AND promised_days IS NOT NULL
    )

    SELECT
        b.*,

        /* ----------------------------------------------------
           ROLLING SELLER HISTORY: PRIOR ORDERS ONLY

           The frame ends at 1 PRECEDING, so the current order
           never contributes to its own feature.

           This prevents target leakage.
        ---------------------------------------------------- */

        AVG(delivery_days) OVER (
            PARTITION BY dominant_seller_id
            ORDER BY purchase_ts
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS seller_hist_days,

        COUNT(*) OVER (
            PARTITION BY dominant_seller_id
            ORDER BY purchase_ts
            ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
        ) AS seller_hist_n

    FROM base b
""")

print("feature view created")

---

In [ ]:
# Load ML Delivery Features

d = q("""
    SELECT *
    FROM `olist.v_ml_delivery_features`
    ORDER BY purchase_ts
""")

# Ensure chronological ordering
d = (
    d.sort_values("purchase_ts")
     .reset_index(drop=True)
)

# Feature Transformations

d["log_price"] = (
    np.log1p(d.item_value)
)

d["log_freight"] = (
    np.log1p(d.freight_value)
)

d["log_weight"] = (
    np.log1p(
        d.total_weight_g.fillna(0)
    )
)


# Category Grouping

top = (
    d.category
    .value_counts()
    .head(15)
    .index
)

d["cat"] = np.where(
    d.category.isin(top),
    d.category,
    "other"
)


# Convert Categorical Columns

CAT_COLS = [
    "cat",
    "customer_state",
    "dominant_seller_state"
]

for c in CAT_COLS:
    d[c] = d[c].astype("category")


# Basic Data Checks

print(
    f"rows: {len(d):,}"
)

print(
    f"date range: "
    f"{d.purchase_date.min()} -> "
    f"{d.purchase_date.max()}"
)

print(
    f"\nseller_hist_days null on first order per seller "
    f"(expected): "
    f"{d.seller_hist_days.isna().sum():,}"
)

**96,470 rows returned.** The `seller_hist_days` nulls are expected: a seller's **first observed order has no prior delivery history** from which to calculate the feature.

These missing values are therefore **structural, not data-quality errors**. The model can retain them as missing; do **not** replace them with a full-period mean, as that would use information from orders outside the prediction point and violate the temporal leakage rules established above.

---

# **4 · Chronological Split**

A random train/test split would allow the model to learn from orders placed **after** the orders it is supposed to predict. This introduces future information into training and can materially inflate evaluation metrics.

The dataset is therefore split **chronologically**, so training uses earlier orders and evaluation is performed on later, unseen orders. This better reflects the conditions the model would face in deployment.

In [ ]:

# Time-Based Train / Test Split

CUTOFF = pd.Timestamp(
    "2018-05-01",
    tz="UTC"
)

d["purchase_ts"] = pd.to_datetime(
    d.purchase_ts,
    utc=True
)

train = d.purchase_ts < CUTOFF
test = d.purchase_ts >= CUTOFF


# Split Summary

print(
    f"train : {train.sum():>7,}   "
    f"{d.purchase_date[train].min()} -> "
    f"{d.purchase_date[train].max()}"
)

print(
    f"test  : {test.sum():>7,}   "
    f"{d.purchase_date[test].min()} -> "
    f"{d.purchase_date[test].max()}"
)


# Delivery Performance: Train vs Test

late = (
    d.delivery_days > d.promised_days
)

print(
    f"\nlate rate  "
    f"train {100 * late[train].mean():.2f}%   "
    f"test {100 * late[test].mean():.2f}%"
)

print(
    f"mean delivery days  "
    f"train {d.delivery_days[train].mean():.1f}   "
    f"test {d.delivery_days[test].mean():.1f}"
)

**71,118 training rows / 25,352 test rows.**

The chronological split creates a clear **distribution shift** between the training and test periods. The late-delivery rate falls from **7.62% in training to 4.40% in test**, while mean delivery time declines from **13.6 to 9.3 days**.

This indicates that fulfilment performance improved over time, so the model is evaluated on a **less difficult regime** than the one represented in training. As a result, absolute test performance may appear somewhat better than it would under a stationary operating environment.

This drift is therefore treated as a **limitation of the evaluation**, not something to correct by randomly reshuffling the data. Relative model comparisons remain informative because all models and baselines are evaluated on the **same held-out test period**.

# **5 · Baselines**

Two baselines are used to establish what level of performance the model must beat.

1. **Naive baseline:** predict the same historical average delivery time for every test order. This measures how much value the model adds beyond a simple constant prediction.

2. **Olist's own checkout estimate:** use the delivery time promised to the customer at checkout as the benchmark. This is particularly valuable because it represents the **real-world estimation system already in use**, rather than an artificial statistical baseline.

The model therefore has to demonstrate improvement over both a simple naive forecast and an **existing operational benchmark**.

In [ ]:
# Baseline Models — Delivery Time Prediction


y = d.delivery_days

y_test = y[test].values


# B1 — Naive Global Mean Baseline

const = y[train].mean()

b1_mae = mean_absolute_error(
    y_test,
    np.full(test.sum(), const)
)

b1_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        np.full(test.sum(), const)
    )
)


# B2 — Olist's Own Promised-Date Estimate

# Olist's promised delivery time as the incumbent
# production estimate
olist_pred = d.promised_days[test].values

b2_mae = mean_absolute_error(
    y_test,
    olist_pred
)

b2_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        olist_pred
    )
)

# Positive bias means Olist tends to overestimate
# delivery time; negative bias means underestimation.
bias = (
    olist_pred - y_test
).mean()


# Display Baseline Results

print(
    f"{'baseline':<32}"
    f"{'MAE':>8}"
    f"{'RMSE':>8}"
)

print(
    f"{'B1  global mean':<32}"
    f"{b1_mae:>8.2f}"
    f"{b1_rmse:>8.2f}"
)

print(
    f"{'B2  Olist own estimate':<32}"
    f"{b2_mae:>8.2f}"
    f"{b2_rmse:>8.2f}"
)

print(
    f"\nOlist bias: {bias:+.2f} days  "
    f"({(olist_pred > y_test).mean() * 100:.1f}% "
    f"of orders arrive early)"
)

**B1 MAE = 6.65 days; B2 MAE = 13.29 days; Olist bias = +12.70 days.**

Olist's own checkout estimate performs **substantially worse than the simple global-mean baseline** on the same test set. However, this does not necessarily indicate a poor estimation system. The **+12.70-day bias**, together with **93.8% of orders arriving earlier than promised**, shows that the estimate is heavily padded.

The estimate therefore appears to function more as a **customer-facing safety buffer than as a point forecast of actual delivery time**.

This reframes the modelling opportunity: the objective is not simply to **beat a weak baseline**, but to determine **how much of Olist's delivery-time buffer is actually necessary** while maintaining reliable customer promises. This question is addressed directly in §8.

---

# **6 · Models**

Three modelling levels are used:

1. **Baselines** — the global-mean forecast and Olist's own checkout estimate, establishing the benchmarks to beat.
2. **Linear regression** — provides an interpretable reference for understanding how the available features relate to delivery time.
3. **Gradient boosting** — the primary predictive model, chosen to capture nonlinear relationships and feature interactions that a linear model may miss.

No additional algorithms are introduced. Beyond these three levels, extra models would add complexity without materially improving the business insight.

In [ ]:
# Feature Sets

FEAT_A = [
    "log_price",
    "log_freight",
    "n_items",
    "log_weight",
    "month",
    "dow"
]

FEAT_B = FEAT_A + [
    "max_distance_km",
    "customer_state",
    "dominant_seller_state",
    "cat"
]

FEAT_C = FEAT_B + [
    "seller_hist_days",
    "seller_hist_n"
]


# Gradient Boosting Regressor Helper

def fit_gbm(
    features,
    target=y,
    seed=RANDOM_STATE
):
    X = d[features]

    cats = [
        X.columns.get_loc(c)
        for c in features
        if c in CAT_COLS
    ]

    m = HistGradientBoostingRegressor(
        max_iter=400,
        learning_rate=0.08,
        random_state=seed,
        categorical_features=cats or None
    )

    m.fit(
        X[train],
        target[train]
    )

    return m, X


# Ridge Regression — Interpretable Linear Reference

X_lin = (
    pd.get_dummies(
        d[FEAT_C],
        columns=CAT_COLS,
        drop_first=True
    )
    .fillna(0)
)

scaler = StandardScaler().fit(
    X_lin[train]
)

ridge = Ridge(
    alpha=1.0
).fit(
    scaler.transform(X_lin[train]),
    y[train]
)

ridge_pred = ridge.predict(
    scaler.transform(X_lin[test])
)


# Model Performance Summary

print(
    f"{'model':<32}"
    f"{'MAE':>8}"
    f"{'RMSE':>8}"
    f"{'vs Olist':>11}"
)

print(
    f"{'B1  global mean':<32}"
    f"{b1_mae:>8.2f}"
    f"{b1_rmse:>8.2f}"
    f"{'':>11}"
)

print(
    f"{'B2  Olist own estimate':<32}"
    f"{b2_mae:>8.2f}"
    f"{b2_rmse:>8.2f}"
    f"{'-':>11}"
)

print(
    f"{'Ridge regression':<32}"
    f"{mean_absolute_error(y_test, ridge_pred):>8.2f}"
    f"{np.sqrt(mean_squared_error(y_test, ridge_pred)):>8.2f}"
    f"{100 * (1 - mean_absolute_error(y_test, ridge_pred) / b2_mae):>10.0f}%"
)


# Gradient Boosting Models

preds = {}

for name, feats in [
    (
        "GBM A  order characteristics",
        FEAT_A
    ),
    (
        "GBM B  + geography",
        FEAT_B
    ),
    (
        "GBM C  + seller history",
        FEAT_C
    )
]:
    m, X = fit_gbm(feats)

    p = m.predict(
        X[test]
    )

    preds[name] = p

    print(
        f"{name:<32}"
        f"{mean_absolute_error(y_test, p):>8.2f}"
        f"{np.sqrt(mean_squared_error(y_test, p)):>8.2f}"
        f"{100 * (1 - mean_absolute_error(y_test, p) / b2_mae):>10.0f}%"
    )


# Final Model — GBM C

model_C, X_C = fit_gbm(
    FEAT_C
)

pred_C = preds[
    "GBM C  + seller history"
]

## **Feature Ablation — What Actually Adds Value?**

The A → B → C progression shows where predictive signal is coming from:

- **A → B: MAE 5.24 → 4.45 days.** Geography provides the largest improvement. Distance and destination-state features account for most of the gain, consistent with Notebook 03's H2, where distance remained strongly associated with delivery time after controlling for other fulfilment variables.

- **B → C: MAE 4.45 → 4.34 days.** Seller history provides an additional but smaller improvement. The gain is real, but modest, so the rolling historical features should be viewed as incremental rather than transformational.

- **Ridge 5.69 vs GBM A 5.24 days.** Both models use the same order-characteristic features, yet GBM performs better. This supports the presence of **nonlinear relationships and/or interactions** that the linear model does not capture, providing a data-driven justification for using gradient boosting as the primary model.

**Final model:** GBM C with seller history achieves **MAE = 4.34 days** and **RMSE = 5.93 days**, reducing MAE by approximately **66% relative to Olist's own estimate** on the same test rows.

## 7. Robustness

The final model is stress-tested beyond its headline MAE to assess **uncertainty, sensitivity to feature choices, and residual behaviour**. A bootstrap confidence interval quantifies uncertainty in test MAE, while adding Olist's own `promised_days` tests whether the model's performance depends on that existing estimate. Residual diagnostics then show where prediction errors remain material.

In [ ]:

# Bootstrap Confidence Interval on Test MAE

rng = np.random.default_rng(
    RANDOM_STATE
)

boot = []

for _ in range(400):
    i = rng.choice(
        len(y_test),
        len(y_test),
        replace=True
    )

    boot.append(
        mean_absolute_error(
            y_test[i],
            pred_C[i]
        )
    )

print(
    f"GBM C test MAE = "
    f"{mean_absolute_error(y_test, pred_C):.2f} days"
)

print(
    f"  bootstrap 95% CI "
    f"[{np.percentile(boot, 2.5):.2f}, "
    f"{np.percentile(boot, 97.5):.2f}]"
)

# Sensitivity Analysis
# Does Adding Olist's Own Estimate as a Feature Help?

FEAT_D = FEAT_C + [
    "promised_days"
]

m_D, X_D = fit_gbm(
    FEAT_D
)

pred_D = m_D.predict(
    X_D[test]
)

print(
    f"\nwith Olist's promised_days as a feature: "
    f"MAE = "
    f"{mean_absolute_error(y_test, pred_D):.2f} days"
)


# Residual Analysis

resid = (
    y_test - pred_C
)

print(
    f"\nresiduals: "
    f"mean {resid.mean():+.2f}  "
    f"median {np.median(resid):+.2f}  "
    f"sd {resid.std():.2f}"
)

print(
    f"  within +/-3 days : "
    f"{100 * (np.abs(resid) <= 3).mean():.1f}%"
)

print(
    f"  within +/-7 days : "
    f"{100 * (np.abs(resid) <= 7).mean():.1f}%"
)

print(
    f"  underestimates by >7 days : "
    f"{100 * (resid > 7).mean():.1f}%"
)

**GBM C MAE = 4.31 days; bootstrap 95% CI [4.26, 4.36].** The narrow interval indicates that the test-set MAE estimate is stable.

Adding Olist's own `promised_days` improves MAE only marginally, from **4.31 to 4.20 days**. The primary model therefore does not depend on the incumbent estimate to achieve strong predictive performance, keeping the comparison against Olist's current system clean.

The residuals are more informative for the operational decision: **84.2% of predictions are within ±7 days**, but **5.6% underestimate actual delivery time by more than 7 days**. These underestimation errors are the cases that require additional buffer in the proposed delivery promise.

---

# **8. The Decision: How Much Padding Is Necessary?**

The model is only valuable if it changes the delivery-promise decision. Olist's current estimate is heavily padded and still misses actual delivery time for a minority of orders. A practical alternative is to set the customer promise as:

$$
\text{Promised Days}
=
\text{Model Prediction}
+
\text{Safety Buffer}
$$

Varying the buffer makes the trade-off explicit: **larger buffers improve reliability but lengthen the promised delivery window**, while smaller buffers provide a tighter promise at the cost of more late deliveries. The objective is therefore to identify the **smallest buffer that delivers an acceptable level of reliability**.

In [ ]:

# Promise Buffer Trade-Off Analysis

rows = []

for buf in [
    0, 2, 3, 4, 5, 6, 8, 10, 12
]:
    promise = pred_C + buf

    rows.append({
        "buffer_days": buf,
        "avg_promise": round(
            promise.mean(),
            1
        ),
        "vs_olist_days": round(
            promise.mean() - olist_pred.mean(),
            1
        ),
        "late_rate_pct": round(
            100 * (y_test > promise).mean(),
            1
        )
    })


# Trade-Off Summary

tradeoff = pd.DataFrame(
    rows
)

print(
    f"Olist today: "
    f"avg promise {olist_pred.mean():.1f} days, "
    f"late rate {100 * (y_test > olist_pred).mean():.1f}%\n"
)

print(
    tradeoff.to_string(
        index=False
    )
)

In [ ]:

# Promise Window vs Late Rate

fig, ax = plt.subplots(
    figsize=(8.5, 5),
    dpi=150
)

# Model predictions with different safety buffers
ax.plot(
    tradeoff.avg_promise,
    tradeoff.late_rate_pct,
    "o-",
    color="#173F73",
    lw=2.4,
    ms=7,
    label="Model + safety buffer"
)

# Current Olist benchmark
ax.scatter(
    [olist_pred.mean()],
    [100 * (y_test > olist_pred).mean()],
    s=180,
    color="#C0392B",
    zorder=5,
    marker="D",
    label="Olist today"
)


# Annotate Safety Buffer Values


for _, r in tradeoff.iterrows():
    ax.annotate(
        f"+{int(r.buffer_days)}d",
        (
            r.avg_promise,
            r.late_rate_pct
        ),
        textcoords="offset points",
        xytext=(0, 9),
        ha="center",
        fontsize=8.5,
        color="#173F73"
    )


# Labels and Title

ax.set_xlabel(
    "Average promised delivery window (days)",
    fontweight="bold"
)

ax.set_ylabel(
    "Late rate (%)",
    fontweight="bold"
)

ax.set_title(
    "Shorter promises and fewer late orders are simultaneously achievable",
    fontweight="bold",
    fontsize=13,
    loc="left"
)


# Styling

ax.grid(
    ls="--",
    alpha=0.4
)

ax.legend(
    frameon=False
)

ax.spines[
    ["top", "right"]
].set_visible(False)

plt.tight_layout()
plt.show()

**Olist's current operating point is dominated by the model-based alternatives.**

- **Model + 8 days:** average promise falls to **18.8 days**, which is **2.2 days shorter** than Olist's current 21.0-day promise, while the late rate falls from **5.5% to 4.6%**. This improves both promise length and reliability simultaneously.

- **Model + 10 days:** average promise is **20.8 days**, essentially matching Olist's current window, but the late rate falls to **3.1%** — a reduction of roughly **44%** relative to 5.5%.

The choice therefore depends on the operating objective: **+8 days prioritises a shorter customer promise**, while **+10 days prioritises additional reliability at approximately the current promise length**.

This connects directly to Notebook 03's customer-experience findings: late delivery is strongly associated with poor reviews. On the 25,352-order test set, reducing the late rate from 5.5% to 3.1% corresponds to roughly **600 fewer late orders** at the same volume.

---

# **9. The Rejected Model**

The natural first approach was to predict `late_order` directly as a classification problem. The classifier was built and evaluated using the **same chronological train/test split** as the regression model.

It was ultimately **rejected because its predictive lift was too limited to support useful operational targeting**. Documenting this failed modelling path is important: it shows that the final regression approach was selected based on **measured business usefulness**, rather than simply choosing the most obvious modelling formulation.

In [ ]:
# Late-Delivery Classification Models

y_late = late.astype(int)

print(
    f"{'model':<30}"
    f"{'PR-AUC':>9}"
    f"{'ROC':>7}"
    f"{'Rec@10%':>10}"
    f"{'Lift@10%':>10}"
)


# Naive Baseline — Prevalence

base_rate = y_late[test].mean()

print(
    f"{'Naive (prevalence)':<30}"
    f"{base_rate:>9.3f}"
    f"{0.500:>7.3f}"
    f"{'10.0%':>10}"
    f"{'1.0x':>10}"
)


# Top-K Recall and Lift

def topk(yt, pp, k=0.10):
    idx = np.argsort(-pp)[
        :int(k * len(pp))
    ]

    return (
        yt[idx].sum() / yt.sum(),
        yt[idx].mean() / yt.mean()
    )


# Test-set target
yt = y_late[test].values


# Gradient Boosting Classifiers

for name, feats in [
    (
        "XGB A  order chars",
        FEAT_A
    ),
    (
        "XGB B  + geography",
        FEAT_B
    ),
    (
        "XGB C  + seller history",
        FEAT_C
    )
]:
    X = d[feats]

    cats = [
        X.columns.get_loc(c)
        for c in feats
        if c in CAT_COLS
    ]

    clf = HistGradientBoostingClassifier(
        max_iter=300,
        learning_rate=0.08,
        random_state=RANDOM_STATE,
        categorical_features=cats or None
    )

    clf.fit(
        X[train],
        y_late[train]
    )

    pp = clf.predict_proba(
        X[test]
    )[:, 1]

    rec, lift = topk(
        yt,
        pp
    )

    print(
        f"{name:<30}"
        f"{average_precision_score(yt, pp):>9.3f}"
        f"{roc_auc_score(yt, pp):>7.3f}"
        f"{rec:>10.1%}"
        f"{lift:>9.1f}x"
    )

    if name.startswith("XGB C"):
        pp_C = pp


# Bootstrap Confidence Interval — Recall@10%

boot_r = []

for _ in range(400):
    i = rng.choice(
        len(yt),
        len(yt),
        replace=True
    )

    # Skip samples containing no positive cases
    if yt[i].sum() == 0:
        continue

    boot_r.append(
        topk(
            yt[i],
            pp_C[i]
        )[0]
    )

print(
    f"\nRecall@10% bootstrap 95% CI: "
    f"[{np.percentile(boot_r, 2.5):.1%}, "
    f"{np.percentile(boot_r, 97.5):.1%}]"
)

**The classifier does not provide enough operational lift to justify deployment.**

The final XGB classifier achieves **PR-AUC = 0.066**, **Recall@10% = 13.8%**, and **Lift@10% = 1.4×**. At a fixed intervention capacity covering the riskiest 10% of orders, the model captures only 13.8% of late deliveries versus 10% under random targeting. The bootstrap 95% CI for Recall@10% is **[12.0%, 15.8%]**, confirming that the lift is measurable but still modest.

The feature progression also points to a clear pattern: **geography provides most of the useful signal**, while seller history adds little additional discrimination. The classifier is therefore not failing because the model family is inadequate; rather, there appears to be limited predictable structure in the `late_order` outcome itself.

This supports the decision to reject the classifier and focus on **predicting delivery time**, where the target contains substantially more systematic signal and can directly support the safety-buffer decision in §8.

# 10 · What Drives the Prediction

Model performance tells us **how well** delivery time can be predicted; feature importance helps explain **why**. This section examines which variables contribute most to the GBM's predictions and whether the model's learned structure is consistent with the statistical relationships identified earlier.

The goal is interpretation, not causality: feature importance shows where the model finds predictive signal, not which factors independently cause longer deliveries.

In [ ]:
# Permutation Feature Importance


pi = permutation_importance(
    model_C,
    X_C[test],
    y_test,
    n_repeats=5,
    random_state=RANDOM_STATE,
    scoring="neg_mean_absolute_error"
)

imp = (
    pd.Series(
        pi.importances_mean,
        index=X_C.columns
    )
    .sort_values(
        ascending=False
    )
)

print(
    "Permutation importance "
    "(MAE increase in days when shuffled):\n"
)

for k, v in imp.items():
    print(
        f"  {k:<26}"
        f"{v:+.3f}"
    )


# Plot Feature Importance

fig, ax = plt.subplots(
    figsize=(8, 4.6),
    dpi=150
)

imp.sort_values().plot.barh(
    ax=ax,
    color="#173F73"
)

ax.set_xlabel(
    "Increase in MAE when shuffled (days)",
    fontweight="bold"
)

ax.set_title(
    "Distance and seller history dominate delivery-time prediction",
    fontweight="bold",
    fontsize=12,
    loc="left"
)

ax.grid(
    axis="x",
    ls="--",
    alpha=0.4
)

ax.spines[
    ["top", "right"]
].set_visible(False)

plt.tight_layout()
plt.show()

**Permutation importance shows which features the model relies on most when predicting delivery time.**

| Feature | MAE increase (days) |
|---|---:|
| `max_distance_km` | +0.261 |
| `seller_hist_days` | +0.245 |
| `customer_state` | +0.165 |
| `log_freight` | +0.105 |
| `dominant_seller_state` | +0.055 |
| ... | ... |
| `month` | −0.014 |

**Permutation importance is used rather than the model's internal split counts**, because split-based importance can favour high-cardinality features such as `customer_state`, which has 27 levels.

Two findings stand out:

- **Distance and seller history are the two strongest features**, with similar importance. Both the order's geographic burden and the seller's prior delivery performance provide substantial predictive signal. This supports retaining the rolling seller-history features despite their additional engineering complexity.

- **`month` has slightly negative importance.** Shuffling it marginally improves MAE, suggesting that month contributes little useful signal beyond the other features. Given the relatively short observation period, there is limited evidence of a stable seasonal pattern. This is retained as a result rather than selectively omitted.

# **11 · Recommendation**

### **Deploy the model to set delivery promises not to flag risky orders.**

The evidence supports two practical operating points:

| Option | Change | Effect |
|---|---|---|
| **Model + 8-day buffer** | Promise **2.2 days shorter** | Late rate **5.5% → 4.6%** |
| **Model + 10-day buffer** | Promise approximately unchanged | Late rate **5.5% → 3.1%** |

Both options dominate Olist's current operating point on the test period. The choice is therefore **commercial rather than purely technical**:

- **+8 days** prioritises a shorter quoted delivery window while still reducing the late rate.
- **+10 days** keeps the average promise approximately unchanged while delivering a larger reliability improvement.

The analysis does **not** establish whether a shorter delivery promise improves conversion, so any conversion benefit remains an untested hypothesis.

### **Why this matters beyond logistics**

Notebook 03 established a strong relationship between delivery lateness and customer dissatisfaction: late orders have substantially higher odds of receiving a **1–2 star review**, and approximately **62% of late orders receive a 1–2 star rating**. Reducing late deliveries is therefore the clearest operational lever in this project with a quantified connection to customer experience.

### **What this will not fix**

The retention analysis provides no evidence that delivery satisfaction is a strong practical driver of repeat purchase. In the final 90-day repeat model, review score has only a **small association with repeat purchase** (OR **0.956**, 95% CI **[0.922, 0.991]**), while the model has very low overall explanatory power.

The observed direction is also counterintuitive, so it should **not** be interpreted as evidence that better reviews cause lower retention. The defensible conclusion is narrower: **improving delivery reliability should improve customer experience, but this analysis does not support expecting it to materially increase retention.**

The marketplace therefore appears to contain **two separate problems**:

> **Fulfilment reliability is actionable through better delivery-time estimation and promise setting; low repeat purchase remains a largely unresolved structural problem.**

# 12 · Limitations

The results are useful for decision support, but several limitations constrain how they should be interpreted and deployed.

**Not a causal estimate.** The buffer analysis is a counterfactual applied to historical orders. It assumes that changing the promised delivery window would not itself change actual delivery behaviour. This is plausible because the promise is customer-facing, but it is not tested here. The recommended buffer should therefore be validated through a controlled rollout or experiment before full deployment.

**Regime drift.** The late-delivery rate falls from **7.62% in training to 4.40% in test**, while average delivery time also declines. Fulfilment performance therefore improved over the observation period, meaning the test set represents an easier operating regime than the training period. Comparisons between models remain fair because all are evaluated on the same held-out test rows, but absolute performance should be re-validated on current data.

**Cold-start sellers.** `seller_hist_days` is null for a seller's first observed order because no prior delivery history exists. Gradient boosting can retain these missing values, but predictions for new sellers rely on geography and order characteristics alone and are therefore expected to be less informative.

**Single time split.** The analysis uses one chronological train/test cutoff rather than rolling-origin cross-validation. The bootstrap confidence interval captures sampling variability in test MAE, but it does not measure sensitivity to alternative cutoff dates or future operating regimes.

**No cost model.** The choice between an 8-day and 10-day buffer ultimately depends on the commercial value of a shorter promise versus the operational and customer cost of a late delivery. Those costs are not available in the dataset, so the analysis identifies viable operating points rather than a single financially optimal policy.

**Distance is approximate.** Seller-customer distance is estimated using zip-prefix geographic information rather than exact addresses. The geolocation data also required deduplication, with approximately **52.6 records per zip prefix** on average. Distance should therefore be interpreted as a geographic proxy rather than an exact route distance.